# BTCUSDT 1h - Strategy Benchmarking, Analytics & Production Verification
Decoupled Analytics Notebook. Evaluates out-of-sample performance, benchmark comparisons, risk metrics, and diagnostic visualizations from trained model predictions in seconds without retraining.

## 1. Setup & Environment

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Running in local environment (Google Colab drive mount skipped).")

In [ ]:
import os

# ============ CONFIG (edit here) ============
TF      = "1h"
SYMBOL  = "BTCUSDT"

DEF_DATA   = "/content/drive/MyDrive/crypto_data" if ("IN_COLAB" in globals() and IN_COLAB) else r"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data"
DEF_MODEL  = "/content/drive/MyDrive/crypto_models" if ("IN_COLAB" in globals() and IN_COLAB) else r"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\model"
DATA_ROOT  = os.environ.get("DATA_ROOT", DEF_DATA)     # data folder
MODEL_ROOT = os.environ.get("MODEL_ROOT", DEF_MODEL)   # model folder

# Execution & Fee Assumptions
FEE_SIDE   = 0.0002                # 0.02% maker fee per side (post-only execution)
SLIP_SIDE  = 0.0001                # 0.01% slippage per side
RISK_PCT   = 0.01                  # risk 1% of equity per trade
MAX_LEV    = 3.0                   # cap on notional leverage
HOLDOUT_MONTHS = 6                 # last N months = sealed hold-out
RANDOM_RUNS    = 300               # Monte Carlo random-entry benchmark simulations
HIGH_CONVICTION_PCT = 0.85         # top 15% probability threshold (cuts fee churn)
BE_TRIGGER_ATR = 0.8               # Kaminski & Lo dynamic breakeven trigger (+0.8 ATR)
BE_SL_ATR      = 0.05              # Ratchet stop (+0.05 ATR covers fees)
# ============================================

## 2. Load Model Artifacts & Out-of-Sample Predictions

In [ ]:
import os, json, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from IPython.display import display

warnings.filterwarnings("ignore")

MODEL_DIR = os.path.join(MODEL_ROOT, TF)
DATA_PATH = os.path.join(DATA_ROOT, TF, f"{SYMBOL}_{TF}.parquet")

# 1. Load OOS predictions
oos_path = os.path.join(MODEL_DIR, "oos_predictions.parquet")
assert os.path.exists(oos_path), f"OOS predictions not found: {oos_path} - run train notebook first."
OOS = pd.read_parquet(oos_path)

# 2. Load model card & hyperparameters
card_path = os.path.join(MODEL_DIR, "model_card.json")
with open(card_path, "r") as f:
    CARD = json.load(f)

# 3. Load market data
assert os.path.exists(DATA_PATH), f"Market data not found: {DATA_PATH} - run download notebook first."
df = pd.read_parquet(DATA_PATH).sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)
df["open_time"] = pd.to_datetime(df["open_time"], utc=True).astype("datetime64[ns, UTC]")
df["close_time"] = pd.to_datetime(df["close_time"], utc=True).astype("datetime64[ns, UTC]")
T_BAR = pd.DatetimeIndex(df["open_time"])

TP_ATR = float(CARD.get("tp_atr", 2.0))
SL_ATR = float(CARD.get("sl_atr", 1.0))
H = int(CARD.get("horizon_bars", 6))
FEATURES = CARD.get("features", [])
COST_SIDE = FEE_SIDE + SLIP_SIDE
DELTA = pd.Timedelta(minutes=15) if TF == "15m" else (pd.Timedelta(hours=1) if TF == "1h" else (pd.Timedelta(hours=4) if TF == "4h" else (pd.Timedelta(days=1) if TF == "1d" else pd.Timedelta(days=7))))

print(f"Loaded {len(OOS):,} OOS prediction bars for {SYMBOL} {TF}")
print(f"Model trained until: {CARD.get('trained_until')} | Mean Long AUC: {CARD.get('mean_auc_long')} | Short AUC: {CARD.get('mean_auc_short')}")
print(f"Triple-barrier settings: TP = {TP_ATR} x ATR, SL = {SL_ATR} x ATR, Horizon = {H} bars")

## 3. Simulator & Performance Metrics Engine

In [ ]:
def atr_wilder(d, n=14):
    pc = d["close"].shift(1)
    tr = pd.concat([d["high"] - d["low"], (d["high"] - pc).abs(), (d["low"] - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0/n, adjust=False, min_periods=n).mean()

def ema(s, n):
    return s.ewm(span=n, adjust=False, min_periods=n).mean()

from numba import njit

@njit(fastmath=True)
def nb_triple_barrier_dynamic(hi, lo, cl, op, a, tp, sl, be_trig, be_sl, Hb):
    n = len(cl)
    m = n - Hb
    RL = np.full(n, np.nan); RS = np.full(n, np.nan)
    yL = np.full(n, np.nan); yS = np.full(n, np.nan)
    eL = np.full(n, -1); eS = np.full(n, -1)
    
    for i in range(m):
        c0 = cl[i]; a0 = a[i]
        if np.isnan(a0) or a0 <= 0: continue
            
        # LONG
        tp_L = c0 + tp * a0; sl_L_orig = c0 - sl * a0; be_L_trig = c0 + be_trig * a0; sl_L_be = c0 + be_sl * a0
        hit_be_L = False; curr_sl_L = sl_L_orig; res_L = False
        for s in range(1, Hb + 1):
            idx = i + s; h, l, o, c = hi[idx], lo[idx], op[idx], cl[idx]
            if c >= o:
                if l <= curr_sl_L:
                    RL[i] = be_sl if hit_be_L else -sl; yL[i] = 1.0 if hit_be_L else 0.0; eL[i] = idx; res_L = True; break
                if not hit_be_L and h >= be_L_trig: hit_be_L = True; curr_sl_L = sl_L_be
                if h >= tp_L: RL[i] = tp; yL[i] = 1.0; eL[i] = idx; res_L = True; break
            else:
                if not hit_be_L and h >= be_L_trig: hit_be_L = True; curr_sl_L = sl_L_be
                if h >= tp_L: RL[i] = tp; yL[i] = 1.0; eL[i] = idx; res_L = True; break
                if l <= curr_sl_L:
                    RL[i] = be_sl if hit_be_L else -sl; yL[i] = 1.0 if hit_be_L else 0.0; eL[i] = idx; res_L = True; break
        if not res_L:
            to_R = (cl[i + Hb] - c0) / a0; RL[i] = to_R; yL[i] = 1.0 if to_R >= 0 else 0.0; eL[i] = i + Hb

        # SHORT
        tp_S = c0 - tp * a0; sl_S_orig = c0 + sl * a0; be_S_trig = c0 - be_trig * a0; sl_S_be = c0 - be_sl * a0
        hit_be_S = False; curr_sl_S = sl_S_orig; res_S = False
        for s in range(1, Hb + 1):
            idx = i + s; h, l, o, c = hi[idx], lo[idx], op[idx], cl[idx]
            if c <= o:
                if h >= curr_sl_S:
                    RS[i] = be_sl if hit_be_S else -sl; yS[i] = 1.0 if hit_be_S else 0.0; eS[i] = idx; res_S = True; break
                if not hit_be_S and l <= be_S_trig: hit_be_S = True; curr_sl_S = sl_S_be
                if l <= tp_S: RS[i] = tp; yS[i] = 1.0; eS[i] = idx; res_S = True; break
            else:
                if not hit_be_S and l <= be_S_trig: hit_be_S = True; curr_sl_S = sl_S_be
                if l <= tp_S: RS[i] = tp; yS[i] = 1.0; eS[i] = idx; res_S = True; break
                if h >= curr_sl_S:
                    RS[i] = be_sl if hit_be_S else -sl; yS[i] = 1.0 if hit_be_S else 0.0; eS[i] = idx; res_S = True; break
        if not res_S:
            to_R = (c0 - cl[i + Hb]) / a0; RS[i] = to_R; yS[i] = 1.0 if to_R >= 0 else 0.0; eS[i] = i + Hb
    return RL, RS, eL, eS

def triple_barrier(d, atr, tp, sl, Hb):
    hi = d["high"].values.astype(np.float64)
    lo = d["low"].values.astype(np.float64)
    cl = d["close"].values.astype(np.float64)
    op = d["open"].values.astype(np.float64)
    a  = atr.values.astype(np.float64)
    be_trig = float(globals().get("BE_TRIGGER_ATR", 0.8))
    be_sl   = float(globals().get("BE_SL_ATR", 0.05))
    return nb_triple_barrier_dynamic(hi, lo, cl, op, a, tp, sl, be_trig, be_sl, Hb)

ATR = atr_wilder(df)
raw_RL, raw_RS, raw_eL, raw_eS = triple_barrier(df, ATR, TP_ATR, SL_ATR, H)

# Mask valid rows
val = np.isfinite(raw_RL) & np.isfinite(raw_eL)
D = df.iloc[val].copy()
D.set_index("open_time", inplace=True)
A_bar = np.arange(len(D))
A_eL = raw_eL[val].astype(int)
A_eS = raw_eS[val].astype(int)
A_RL = (raw_RL[val] - COST_SIDE * (1 + TP_ATR)) / (1.0 + COST_SIDE)
A_RS = (raw_RS[val] - COST_SIDE * (1 + TP_ATR)) / (1.0 + COST_SIDE)
A_scale = np.minimum(1.0 / (SL_ATR * ATR[val].values / D["close"].values), MAX_LEV)

# Benchmarks: Donchian 20 breakout
D["don_long"]  = (D["close"] > D["high"].shift(1).rolling(20).max()).values
D["don_short"] = (D["close"] < D["low"].shift(1).rolling(20).min()).values

PERIOD = "W" if TF == "1w" else "D"
ANN = 52 if TF == "1w" else 365

def simulate(pos_rows, sigL, sigS, scL, scS):
    take = np.flatnonzero(sigL | sigS)
    busy, rec = -1, []
    for q in take:
        p = pos_rows[q]
        if A_bar[p] < busy:
            continue
        if sigL[q] and sigS[q]:
            s = 1 if scL[q] >= scS[q] else -1
        else:
            s = 1 if sigL[q] else -1
        e = A_eL[p] if s == 1 else A_eS[p]
        R = A_RL[p] if s == 1 else A_RS[p]
        rec.append((p, s, R, e)); busy = e
    tr = pd.DataFrame(rec, columns=["pos","side","R","exit_bar"])
    if len(tr):
        tr["entry_time"] = D.index[tr["pos"].values]
        tr["exit_time"]  = T_BAR[tr["exit_bar"].values]
        tr["ret"] = RISK_PCT * tr["R"].values * A_scale[tr["pos"].values]
    return tr

def period_returns(tr, t0, t1):
    idx = pd.date_range(t0.floor("D"), t1.ceil("D"), freq=PERIOD, tz="UTC")
    r = pd.Series(0.0, index=idx)
    if tr is not None and len(tr):
        ts = tr.set_index("exit_time")["ret"].resample(PERIOD).sum()
        r = r.add(ts, fill_value=0.0)
    else:
        c = df.set_index("open_time")["close"].resample(PERIOD).last().loc[t0:t1]
        r = c.pct_change().fillna(0.0)
    return r

def compute_metrics(r, tr, t0, t1):
    m = {}
    eq = (1 + r).cumprod()
    days = max(1, (t1 - t0).total_seconds() / 86400)
    m["total_return"] = eq.iloc[-1] - 1
    m["cagr"] = (eq.iloc[-1] ** (365.25 / days)) - 1 if eq.iloc[-1] > 0 else -1.0
    s = r.std()
    m["sharpe"] = (r.mean() / s * np.sqrt(ANN)) if s > 1e-12 else np.nan
    ds = r[r < 0].std()
    m["sortino"] = (r.mean() / ds * np.sqrt(ANN)) if ds > 1e-12 else np.nan
    peak = eq.cummax()
    dd = (eq - peak) / peak
    m["max_dd"] = dd.min()
    m["calmar"] = (m["cagr"] / abs(m["max_dd"])) if abs(m["max_dd"]) > 1e-6 else np.nan
    if tr is not None and len(tr):
        m["trades"] = len(tr)
        m["win_rate"] = (tr["R"] > 0).mean()
        m["avg_R"] = tr["R"].mean()
        gw = tr.loc[tr["R"] > 0, "R"].sum()
        gl = abs(tr.loc[tr["R"] < 0, "R"].sum())
        m["profit_factor"] = (gw / gl) if gl > 1e-6 else np.nan
        held = sum((x - e).total_seconds() for e, x in zip(tr["entry_time"], tr["exit_time"]))
        m["exposure"] = held / (days * 86400)
    else:
        m.update(trades=np.nan, win_rate=np.nan, avg_R=np.nan, profit_factor=np.nan, exposure=1.0)
    return m

## 4. Multi-Strategy Benchmarking (OOS / DEV / Sealed Hold-Out)

In [ ]:
P = OOS["pos"].values
OOS_T = D.index[P]
HOLDOUT_START = T_BAR[-1] - pd.DateOffset(months=HOLDOUT_MONTHS)

sigL_ml = OOS["pL"].values >= OOS["thrL"].values
sigS_ml = OOS["pS"].values >= OOS["thrS"].values
scL, scS = OOS["pL"].values - OOS["thrL"].values, OOS["pS"].values - OOS["thrS"].values

# 1. Macro Trend Filter (Price vs 50 EMA on higher timeframe / 120 bars on current)
macro_trend = ema(D["close"], 120).values[P]
macro_bull = D["close"].values[P] >= macro_trend
macro_bear = D["close"].values[P] <= macro_trend

# 2. High-Conviction Thresholding (top 15% probabilities to eliminate fee churn)
top_pct = HIGH_CONVICTION_PCT
qL_high = np.quantile(OOS["pL"], top_pct)
qS_high = np.quantile(OOS["pS"], top_pct)
sigL_conv = OOS["pL"].values >= qL_high
sigS_conv = OOS["pS"].values >= qS_high

# 3. Meta-Labeling (Donchian Trend Breakout + ML Quality Confirmation)
don_L = D["don_long"].values[P]
don_S = D["don_short"].values[P]
sigL_meta = don_L & (OOS["pL"].values >= np.median(OOS["pL"]))
sigS_meta = don_S & (OOS["pS"].values >= np.median(OOS["pS"]))

# 4. Sniper 85%+ Target (Top 5% highest confidence + Macro Trend)
qL_sniper = np.quantile(OOS["pL"], 0.95)
qS_sniper = np.quantile(OOS["pS"], 0.95)
sigL_sniper = (OOS["pL"].values >= qL_sniper) & macro_bull
sigS_sniper = (OOS["pS"].values >= qS_sniper) & macro_bear

# Logistic Regression baseline
thrLq = np.quantile(OOS["qL"], 1 - max(sigL_ml.mean(), 1e-6))
thrSq = np.quantile(OOS["qS"], 1 - max(sigS_ml.mean(), 1e-6))
sigL_lr = OOS["qL"].values >= thrLq
sigS_lr = OOS["qS"].values >= thrSq
n = len(P); ZERO = np.zeros(n)

VARIANTS = {
    "ML Sniper 85%+ Target":           (sigL_sniper, sigS_sniper, scL, scS),
    "ML + Macro Trend (Recommended)": (sigL_ml & macro_bull, sigS_ml & macro_bear, scL, scS),
    "ML High-Conviction + Trend":      (sigL_conv & macro_bull, sigS_conv & macro_bear, scL, scS),
    "Donchian Meta-Labeling":          (sigL_meta, sigS_meta, scL, scS),
    "ML only (no rules)":              (sigL_ml, sigS_ml, scL, scS),
    "LogReg Baseline":                 (sigL_lr, sigS_lr, OOS["qL"].values - thrLq, OOS["qS"].values - thrSq),
    "Donchian-20 breakout":            (don_L, don_S, ZERO, ZERO),
    "Always-long (barriers)":          (np.ones(n, bool), np.zeros(n, bool), ZERO, ZERO),
}
WINDOWS = {"ALL OOS": np.ones(n, bool), "DEV (walk-forward)": OOS_T < HOLDOUT_START, "SEALED HOLD-OUT": OOS_T >= HOLDOUT_START}

def run_variant(name, win):
    rows = np.flatnonzero(win)
    if len(rows) == 0: return None, None, None, None
    sL, sS, a, b = VARIANTS[name]
    trd = simulate(P[rows], sL[rows], sS[rows], a[rows], b[rows])
    t0, t1 = OOS_T[rows[0]], OOS_T[rows[-1]] + DELTA
    r = period_returns(trd, t0, t1)
    return compute_metrics(r, trd, t0, t1), trd, r, (t0, t1)

M, TRADES, EQ = {}, {}, {}
for wname, win in WINDOWS.items():
    for vname in VARIANTS:
        m, trd, r, span = run_variant(vname, win)
        if m is None: continue
        M[(wname, vname)] = m
        if wname == "ALL OOS": TRADES[vname], EQ[vname] = trd, r
    rows = np.flatnonzero(win)
    if len(rows):
        t0, t1 = OOS_T[rows[0]], OOS_T[rows[-1]] + DELTA
        r = period_returns(None, t0, t1)
        M[(wname, "Buy & Hold")] = compute_metrics(r, None, t0, t1)
        if wname == "ALL OOS": EQ["Buy & Hold"] = r

RES = pd.DataFrame(M).T
RES.index.names = ["window", "strategy"]
cols = ["total_return","cagr","sharpe","sortino","max_dd","calmar","trades","win_rate","avg_R","profit_factor","exposure"]
for wname in WINDOWS:
    if wname in RES.index.get_level_values(0):
        print(f"\n=== {wname} ===")
        display(RES.loc[wname][cols].round(3))
RES[cols].to_csv(os.path.join(MODEL_DIR, "benchmark_metrics.csv"))
if "ML + Macro Trend (Recommended)" in TRADES and len(TRADES["ML + Macro Trend (Recommended)"]):
    TRADES["ML + Macro Trend (Recommended)"].to_csv(os.path.join(MODEL_DIR, "trades_main.csv"), index=False)

## 5. Statistical Significance (Monte Carlo & Bootstrap)

In [ ]:
rec_name = "ML + Macro Trend (Recommended)"
main_tr = TRADES.get(rec_name, pd.DataFrame())
ml_all = RES.loc[("ALL OOS", rec_name)] if ("ALL OOS", rec_name) in RES.index else RES.iloc[0]

n_tr = len(main_tr) if len(main_tr) else 100
t0, t1 = OOS_T[0], OOS_T[-1] + DELTA

# Monte Carlo simulation
rand_sharpe = []
rng = np.random.default_rng(42)
for _ in range(RANDOM_RUNS):
    picks = rng.choice(n, size=n_tr, replace=False)
    sL_r, sS_r = np.zeros(n, bool), np.zeros(n, bool)
    sides = rng.choice([1, -1], size=n_tr)
    sL_r[picks[sides == 1]] = True
    sS_r[picks[sides == -1]] = True
    tr_r = simulate(P, sL_r, sS_r, ZERO, ZERO)
    r_r = period_returns(tr_r, t0, t1)
    s_ = r_r.std()
    rand_sharpe.append((r_r.mean() / s_ * np.sqrt(ANN)) if s_ > 1e-12 else 0.0)
rand_sharpe = np.array(rand_sharpe)

p_beat = (rand_sharpe >= ml_all["sharpe"]).mean()
print(f"Random-entry benchmark ({RANDOM_RUNS} runs, same signal rate, same barriers & costs):")
print(f"  random Sharpe median {np.median(rand_sharpe):.2f} | 95th pct {np.percentile(rand_sharpe, 95):.2f} | Strategy Sharpe {ml_all['sharpe']:.2f} | P(random >= Strategy) = {p_beat:.3f}")

if len(main_tr):
    R_vals = main_tr["R"].values
    boot_means = [np.mean(rng.choice(R_vals, size=len(R_vals), replace=True)) for _ in range(2000)]
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
    p_neg = (np.array(boot_means) <= 0).mean()
    print(f"Mean R per trade (net of costs) = {R_vals.mean():+.3f}  | 95% bootstrap CI [{ci_lo:+.3f}, {ci_hi:+.3f}]  | P(mean<=0) = {p_neg:.3f}")

## 6. Performance Visualizations & Diagnostic Plots

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(19, 9))

# 1. Equity Curves
for name, r in EQ.items():
    eq = (1 + r).cumprod()
    ax[0, 0].plot(eq.index, eq.values, label=name, lw=2.2 if "Recommended" in name or "Meta" in name else 1.2, alpha=1 if "Recommended" in name or "Hold" in name else .75)
ax[0, 0].axvline(HOLDOUT_START, color="k", ls="--", lw=1)
ax[0, 0].set_yscale("log"); ax[0, 0].set_title("Out-of-sample equity (net of costs, log scale)"); ax[0, 0].legend(fontsize=7)

# 2. Drawdowns
for name in [rec_name, "Buy & Hold"]:
    if name in EQ:
        e = (1 + EQ[name]).cumprod(); ax[0, 1].plot(e.index, (e / e.cummax() - 1).values * 100, label=name)
ax[0, 1].set_title("Drawdown (%)"); ax[0, 1].legend(fontsize=8)

# 3. Model AUC Ranking
auc_csv = os.path.join(MODEL_DIR, "model_comparison_auc.csv")
if os.path.exists(auc_csv):
    auc_df = pd.read_csv(auc_csv)
    mean_aucs = auc_df.drop(columns=["fold"], errors="ignore").mean().sort_values()
    ax[0, 2].barh(mean_aucs.index, mean_aucs.values, color="teal", alpha=0.8)
    ax[0, 2].axvline(0.5, color="r", ls="--")
    ax[0, 2].set_title("Out-of-sample AUC by Model (0.50 = random)")

# 4. Monthly Returns
if rec_name in EQ:
    try:
        monthly = EQ[rec_name].resample("ME").sum() * 100
    except Exception:
        monthly = EQ[rec_name].resample("M").sum() * 100
    pos = monthly >= 0
    ax[1, 0].bar(monthly.index, monthly.values, width=25, color=np.where(pos, "green", "red"), alpha=0.7)
    ax[1, 0].set_title(f"Monthly Returns (%) - {rec_name}")

# 5. Trade R Distribution
if len(main_tr):
    ax[1, 1].hist(main_tr["R"], bins=25, color="royalblue", alpha=0.7)
    ax[1, 1].axvline(0, color="k", ls="--")
    ax[1, 1].axvline(main_tr["R"].mean(), color="red", ls="-", label=f"Mean: {main_tr['R'].mean():+.2f}R")
    ax[1, 1].set_title(f"Trade R Distribution ({rec_name})")
    ax[1, 1].legend(fontsize=8)

# 6. Monte Carlo Sharpe Distribution
if len(rand_sharpe):
    ax[1, 2].hist(rand_sharpe[~np.isnan(rand_sharpe)], bins=30, alpha=.7, label="random entries")
    ax[1, 2].axvline(ml_all["sharpe"], color="r", lw=2, label="Strategy Sharpe")
    ax[1, 2].legend()
ax[1, 2].set_title("Sharpe: Strategy vs Random-Entry Distribution")

plt.tight_layout()
report_path = os.path.join(MODEL_DIR, "report_benchmarks.png")
plt.savefig(report_path, dpi=120)
plt.show()
print(f"Saved analytics diagnostic plot -> {report_path}")

## 7. Production Specification Verdict

In [ ]:
g = lambda w, s, k: RES.loc[(w, s), k] if (w, s) in RES.index else np.nan

checks = [
    ("Mean AUC (long & short) > 0.52", float(CARD.get("mean_auc_long", 0.5)) > 0.52 and float(CARD.get("mean_auc_short", 0.5)) > 0.52),
    ("OOS Sharpe > Buy & Hold Sharpe", g("ALL OOS", rec_name, "sharpe") > g("ALL OOS", "Buy & Hold", "sharpe")),
    ("OOS Sharpe beats 95% of random-entry runs", p_beat < 0.05),
    ("Mean R/trade > 0 (bootstrap p < 0.05)", (p_neg < 0.05) if len(main_tr) else False),
    ("ML beats LogReg baseline (OOS Sharpe)", g("ALL OOS", rec_name, "sharpe") > g("ALL OOS", "LogReg Baseline", "sharpe")),
    ("Sealed hold-out Sharpe > 0", g("SEALED HOLD-OUT", rec_name, "sharpe") > 0),
    ("At least 20 OOS trades", len(main_tr) >= 20),
]

print(f"VERDICT for {SYMBOL} {TF} ({rec_name})")
for title, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {title}")
n_pass = sum(int(ok) for _, ok in checks)
print(f"\n{n_pass}/{len(checks)} checks passed.")

## 8. Interactive Strategy Sandbox (Instant Backtesting)
Tweak probability thresholds, filters, or barriers below and see the updated P&L and metrics in **under 1 second** without re-training!

In [ ]:
# --- Sandbox Controls (edit and re-run this cell) ---
SANDBOX_CONVICTION_QUANTILE = 0.85   # e.g., 0.80, 0.85, 0.90 (top quantile)
USE_MACRO_TREND_FILTER     = True   # only longs in macro uptrend
ALLOW_SHORTS                = False  # in crypto bull cycles, disabling shorts often boosts Sharpe!
CUSTOM_RISK_PCT             = 0.01   # 1% equity risk per trade

# Generate sandbox signals
q_thr_L = np.quantile(OOS["pL"], SANDBOX_CONVICTION_QUANTILE)
q_thr_S = np.quantile(OOS["pS"], SANDBOX_CONVICTION_QUANTILE)

sbx_sigL = OOS["pL"].values >= q_thr_L
sbx_sigS = OOS["pS"].values >= q_thr_S

if USE_MACRO_TREND_FILTER:
    sbx_sigL &= macro_bull
    sbx_sigS &= macro_bear

if not ALLOW_SHORTS:
    sbx_sigS = np.zeros(len(P), bool)

tr_sbx = simulate(P, sbx_sigL, sbx_sigS, scL, scS)
if len(tr_sbx):
    r_sbx = period_returns(tr_sbx, t0, t1)
    m_sbx = compute_metrics(r_sbx, tr_sbx, t0, t1)
    
    print("=== Sandbox Backtest Results ===")
    print(f"Total Return: {m_sbx['total_return']:+.2%} | CAGR: {m_sbx['cagr']:+.2%} | Sharpe: {m_sbx['sharpe']:.2f} | Max DD: {m_sbx['max_dd']:.2%}")
    print(f"Trades: {m_sbx['trades']:.0f} | Win Rate: {m_sbx['win_rate']:.1%} | Profit Factor: {m_sbx['profit_factor']:.2f} | Avg R: {m_sbx['avg_R']:+.2f}R")
    
    # Plot Sandbox Equity Curve
    plt.figure(figsize=(10, 4))
    plt.plot((1 + r_sbx).cumprod(), label="Sandbox Strategy", color="purple", lw=2)
    plt.plot((1 + EQ["Buy & Hold"]).cumprod(), label="Buy & Hold", color="gray", alpha=0.6, ls="--")
    plt.title("Sandbox Strategy vs Buy & Hold (Log Scale)")
    plt.yscale("log"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No trades triggered with current sandbox settings.")